In [1]:
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent

import pandas as pd

# pd.set_option('display.max_columns', None)  # none은 제한값 자리(몇개 열까지 보여줄까)
# pd.set_option('display.max_rows', None)

# ============================================================
# 0. 데이터 불러오기
# ============================================================

train_full = pd.read_parquet(PROJECT_ROOT / 'data' / '2_train_full_clean.parquet')
test = pd.read_parquet(PROJECT_ROOT / 'data' / '2_test_clean.parquet')

print(train_full.shape, test.shape)

(749609, 34) (186263, 33)


## 3. 강우사상 기준 데이터 분할

In [2]:
# ============================================================
# 3-1. 강우사상별 데이터 분할 -> event_id 컬럼 생성
# ============================================================

# 전체 timestamp 추출(중복없이), 시간순 정렬
event_time = (
    train_full[['timestamp']] 
    .drop_duplicates() 
    .sort_values('timestamp') 
    .reset_index(drop = True) 
)

# 차분
event_time['event_time_diff'] = (
    event_time['timestamp']
    .diff()
)

# 차분값이 1시간이 아닌 경우를 event 경계로 추출
cond_split_time = (
    event_time['event_time_diff'] != pd.Timedelta(hours=1)
)

# 강우사상 시간대별 event_id 부여
event_time['event_id'] = cond_split_time.cumsum()

# 시험처가 제시한 강우사상 개수와 맞는지 확인: 122개
assert event_time['event_id'].nunique() == 122, (
    f"주최측이 제시한 강우사상 122개, 현재 {event_time['event_id'].nunique()}개, 오류"
)

# train_full에 event_id 컬럼 추가(매핑 기준열: 'timestamp')
rule_event_id_map = event_time.set_index('timestamp')['event_id']  # 매핑규칙

train_full['event_id'] = train_full['timestamp'].map(rule_event_id_map)

assert not train_full['event_id'].isnull().any(), (
    'event_id 매핑 누락 발생'
)

In [ ]:
# ============================================================
# 3-2. 강우사상별 데이터 분할
# ============================================================

split_num = int((train_full['event_id'].nunique())*0.8) + 1
print('강우사상 분리 기준번호: ', split_num)

train = train_full.loc[train_full['event_id'] <= split_num].copy()
val = train_full.loc[train_full['event_id'] > split_num].copy()

assert train['event_id'].nunique() == split_num, 'Train 분할 이상'
assert train['event_id'].nunique() + val['event_id'].nunique() == train_full['event_id'].nunique(), '전체 강우사상 개수 불일치'

강우사상 분리 기준번호:  98


In [ ]:
# ============================================================
# 3-3. train, val 분리 저장
# ============================================================

train.to_parquet(PROJECT_ROOT / 'data' / '3_train.parquet')
val.to_parquet(PROJECT_ROOT / 'data' / '3_val.parquet')

print("Parquet 저장완료")

Parquet 저장완료


## train데이터 5-fold 분할하기

In [ ]:
# ============================================================
# 4. GroupKFold 5-Fold 배정, parquet 저장
# ============================================================

from sklearn.model_selection import GroupKFold

# ------------------------------------------------------------
# Group 설정: 강우사상(event_id) 단위로 fold 배정
# ------------------------------------------------------------

kf_groups = train['event_id']

print('<시간사상 Group 확인>')
print(f'전체 학습 행 수   : {len(train):,}')  # :,는 천단위 쉼표 넣는 기능
print(f'시간사상 개수     : {kf_groups.nunique():,}')
print()

assert not kf_groups.isnull().any(), 'event_id 결측치 존재'


# ------------------------------------------------------------
# GroupKFold 정의
# ------------------------------------------------------------

N_SPLITS = 5

gkf = GroupKFold(n_splits=N_SPLITS)
# shuffle=False(기본값), Fold별 행 개수가 비슷하게 배정
# 결과물: fold번호, (4Fold idx, 1Fold idx)

# 강우사상별 Fold 번호 저장용 데이터프레임
kf_fold_f = pd.DataFrame({
    'event_id': kf_groups.unique()
})

kf_fold_f['fold_num'] = 0  # Fold 번호 컬럼 생성


# ------------------------------------------------------------
# Fold 배정 및 검증
# ------------------------------------------------------------

for fold, (train_idx, valid_idx) in enumerate(  
    gkf.split(train, groups=kf_groups),  # groups: 같은 Fold에 묶어야 할 그룹 정보
    start=1  # fold 1부터 시작
):
    # Fold별 train, val 사상 중복 여부 확인
    train_groups = set(kf_groups.iloc[train_idx])  # kf_groups = train['event_id'] 이므로
    valid_groups = set(kf_groups.iloc[valid_idx])  # event_id 기준으로 중복이 제거된 집합이 됨
    overlap = train_groups & valid_groups
    assert len(overlap) == 0, f'Fold {fold}: train, val 간에 사상 중복'

    # 검증 Fold에 해당하는 event_id에 Fold 번호 부여
    kf_fold_f.loc[
        kf_fold_f['event_id'].isin(valid_groups),  # 'event_id'별로 valid group에 포함된 경우
        'fold_num'  # 'fold_num' 컬럼에
    ] = fold  # fold번호를 값으로 넣음

    print(
        f'Fold {fold} | '
        f'train={len(train_idx):,} | '
        f'valid={len(valid_idx):,} | '
        f'train events={len(train_groups):,} | '
        f'valid events={len(valid_groups):,}'
    )


# ------------------------------------------------------------
# 3-4. Fold 배정 결과 최종 검증
# ------------------------------------------------------------

assert kf_fold_f['event_id'].is_unique, 'event_id 중복'
assert kf_fold_f['fold_num'].between(1, N_SPLITS).all(), 'Fold 배정 누락'


# ------------------------------------------------------------
# 3-5. Fold 배정 parquet 저장
# ------------------------------------------------------------

kf_fold_f.to_parquet(
    PROJECT_ROOT / 'data' / '0_kf_train_match.parquet',
    index=False
)

print('\nFold 배정 parquet 저장 완료')


<시간사상 Group 확인>
전체 학습 행 수   : 616,293
시간사상 개수     : 98

Fold 1 | train=493,297 | valid=122,996 | train events=82 | valid events=16
Fold 2 | train=492,894 | valid=123,399 | train events=77 | valid events=21
Fold 3 | train=492,894 | valid=123,399 | train events=77 | valid events=21
Fold 4 | train=493,077 | valid=123,216 | train events=78 | valid events=20
Fold 5 | train=493,010 | valid=123,283 | train events=78 | valid events=20

Fold 배정 parquet 저장 완료
